## 多代理系统
### 为什么需要多代理系统？
在现实世界中，许多任务需要多个步骤和不同的技能来完成。单一代理可能无法有效地处理所有这些任务，因为它可能缺乏特定领域的知识或工具。多代理系统通过创建多个专注于不同任务的子代理，可以更高效地完成复杂的任务。每个子代理可以专注于特定的领域或工具，从而提高整体系统的性能和灵活性。

另一个重要原因是如果提供给单一代理的工具过多，可能会导致它难以选择正确的工具来完成任务。通过将工具分配给不同的子代理，可以减少每个代理需要处理的工具数量，从而提高它们的效率和准确性。

### 多代理系统的实现
多代理系统的基本思想是创建多个子代理，每个子代理专注于特定的任务或工具。主代理可以根据需要调用不同的子代理来完成复杂的任务。
在这个示例中，我们将创建一个主代理和两个子代理。第一个子代理负责计算一个数的平方，第二个子代理负责计算一个数的平方根。
主代理将根据用户的输入决定调用哪个子代理来完成任务，具体来说，主代理将两个子代理作为工具使用。

In [10]:
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage
from langchain.tools import tool
from langchain_deepseek import ChatDeepSeek

from dotenv import load_dotenv
import os
load_dotenv()

True

In [11]:
# 初始化语言模型

llm_ds = ChatOpenAI(
    model='my_local_model',
    base_url='http://127.0.0.1:1234/v1',
    api_key='none_for_need',
)

In [12]:
# 定义工具函数

@tool
def square(x: int) -> int:
    """计算一个数的平方"""
    return x * x


@tool
def square_root(x: int) -> float:
    """计算一个数的平方根"""
    return x ** 0.5


# 创建两个子代理，每个子代理使用不同的工具
subagent_1 = create_agent(llm_ds, tools=[square])
subagent_2 = create_agent(llm_ds, tools=[square_root])


In [13]:
# 创建主代理，主代理可以调用两个子代理

@tool
def call_subagent_1(x: int) -> dict:
    """调用子代理1计算平方"""
    responses_agent = subagent_1.invoke({'messages': HumanMessage(content=f"计算 {x} 的平方")})
    return {'messages': responses_agent['messages'][-1].content}


@tool
def call_subagent_2(x: int) -> dict:
    """调用子代理2计算平方根"""
    responses_agent = subagent_2.invoke({'messages':HumanMessage(content= f"计算 {x} 的平方根")})
    return {'messages': responses_agent['messages'][-1].content}

main_agent = create_agent(llm_ds, tools=[call_subagent_1, call_subagent_2])

In [18]:
# 测试主代理

res = main_agent.invoke({'messages':HumanMessage(content="请计算16的平方和81的平方根,给出详细的步骤和结果。分别调用两个子代理来完成这两个任务。")})


In [19]:
print(res['messages'][-1].content)

下面是详细的计算步骤和结果：

**1. 计算16的平方**

*   **调用子代理1的结果：** 子代理1计算得出 $16^2 = 256$。
*   **步骤：** 将数字16乘以自身，得到 256。

**2. 计算81的平方根**

*   **调用子代理2的结果：** 子代理2计算得出 $\sqrt{81} = 9$。
*   **步骤：** 找到一个数，该数的平方等于81，即 $9 \times 9 = 81$。

---

**最终结果汇总：**

*   16的平方是 **256**。
*   81的平方根是 **9**。
